# NB00 — Q1 Diagnosis and Decision Gates

This notebook is the **diagnosis-first entry point**. It answers these questions before any serious model training:

1. Is the dataset schema valid?
2. Are train / test / OOD splits usable?
3. Are control anchors reliable for each cell type?
4. How strong are trivial baselines?
5. Is this dataset suitable for a control-anchored residual model?

**What to send back after running**
- printed dataset summary
- split counts
- control counts by cell type
- baseline metrics table
- decision-gate messages


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import urllib.request
from pathlib import Path
import random

import anndata as ad
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Define paths
BASE_DIR = Path("/content/drive/MyDrive/ChemDFM")
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results_nb00"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "sciplex_complete_middle_subset.h5ad"

# Download if missing
if not DATA_PATH.exists():
    print("Downloading SciPlex dataset (this may take a few minutes)...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    try:
        urllib.request.urlretrieve(URL, DATA_PATH)
        print(f"Download complete! Saved to: {DATA_PATH}")
    except Exception as e:
        print(f"Download failed: {e}")
        raise
else:
    print(f"Dataset already exists at: {DATA_PATH}")

# Config
class CFG:
    def __init__(self):
        self.data_path = str(DATA_PATH)
        self.results_dir = str(RESULTS_DIR)
        self.split_col = "split_ho_pathway"
        self.pca_dim = 25
        self.condition_col = "condition"
        self.cell_col = "cell_type"
        self.dose_col = "dose"
        self.fallback_dose_col = "dose_val"

cfg = CFG()
print(cfg)

Download complete! Saved to: /content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad


In [ ]:
adata = ad.read_h5ad(cfg.data_path)
print(adata)
print("\nobs columns:")
print(list(adata.obs.columns))
print("\nShape:", adata.shape)

AnnData object with n_obs × n_vars = 354640 × 2000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'condition', 'drug_dose_name', 'cov_drug_dose_name', 'cov_drug', 'control', 'split_ho_pathway', 'split_tyrosine_ood', 'split_epigenetic_ood', 'split_cellcycle_ood', 'SMILES', 'split_ood_finetuning', 'split_ho_epigenetic', 'split_ho_epigenetic_all', 'split_random'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'gene_id', 'in_lincs', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'all_DEGs', 'hvg', 'lincs_DEGs', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

obs columns:
['cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_sc

In [ ]:
adata = ad.read_h5ad(cfg.data_path)
print(adata)
print("\nobs columns:")
print(list(adata.obs.columns))

if cfg.dose_col not in adata.obs.columns and cfg.fallback_dose_col in adata.obs.columns:
    adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose_col]

required = [cfg.condition_col, cfg.cell_col, cfg.dose_col, cfg.split_col]
for c in required:
    if c not in adata.obs.columns:
        raise ValueError(f"Missing required column: {c}")

AnnData object with n_obs × n_vars = 354640 × 2000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'condition', 'drug_dose_name', 'cov_drug_dose_name', 'cov_drug', 'control', 'split_ho_pathway', 'split_tyrosine_ood', 'split_epigenetic_ood', 'split_cellcycle_ood', 'SMILES', 'split_ood_finetuning', 'split_ho_epigenetic', 'split_ho_epigenetic_all', 'split_random'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'gene_id', 'in_lincs', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'all_DEGs', 'hvg', 'lincs_DEGs', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

obs columns:
['cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_sc

In [ ]:
def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

adata.obs["_split3"] = adata.obs[cfg.split_col].astype(str).map(map_split)
adata = adata[adata.obs["_split3"].isin(["train","test","ood"])].copy()

print("Split counts:")
print(adata.obs["_split3"].value_counts())

summary = {
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "n_conditions": int(adata.obs[cfg.condition_col].nunique()),
    "n_cell_types": int(adata.obs[cfg.cell_col].nunique()),
    "n_controls": int((adata.obs[cfg.condition_col].astype(str) == "control").sum()),
}
print("\nDataset summary:")
for k, v in summary.items():
    print(f"{k}: {v}")

pd.DataFrame([summary]).to_csv(f"{cfg.results_dir}/dataset_summary.csv", index=False)
adata.obs.groupby(["_split3", cfg.cell_col]).size().rename("count").reset_index().to_csv(
    f"{cfg.results_dir}/split_by_celltype.csv", index=False
)
adata.obs.groupby(["_split3", cfg.condition_col]).size().rename("count").reset_index().to_csv(
    f"{cfg.results_dir}/split_by_condition.csv", index=False
)

Split counts:
_split3
train    292283
test      51798
ood       10559
Name: count, dtype: int64

Dataset summary:
n_cells: 354640
n_genes: 2000
n_conditions: 188
n_cell_types: 3
n_controls: 13004


/tmp/ipykernel_5859/621374715.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(["_split3", cfg.cell_col]).size().rename("count").reset_index().to_csv(
/tmp/ipykernel_5859/621374715.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(["_split3", cfg.condition_col]).size().rename("count").reset_index().to_csv(


In [ ]:
X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)

train_mask = (adata.obs["_split3"] == "train").values
pca = PCA(n_components=cfg.pca_dim, random_state=SEED)
X_pca = np.zeros((adata.n_obs, cfg.pca_dim), dtype=np.float32)
X_pca[train_mask] = pca.fit_transform(X[train_mask]).astype(np.float32)
X_pca[~train_mask] = pca.transform(X[~train_mask]).astype(np.float32)

print("PCA explained variance sum:", float(pca.explained_variance_ratio_.sum()))
np.save(f"{cfg.results_dir}/X_pca_{cfg.pca_dim}d.npy", X_pca)

PCA explained variance sum: 0.2873401939868927


In [ ]:
cell_enc = LabelEncoder()
cell_enc.fit(adata.obs[cfg.cell_col].astype(str))
idx2cell = {i:c for i,c in enumerate(cell_enc.classes_)}
print("Cell types:", idx2cell)

control_mask = (adata.obs[cfg.condition_col].astype(str) == "control").values
ctrl_means = {}
ctrl_counts = {}
for cell in sorted(adata.obs[cfg.cell_col].astype(str).unique()):
    m = control_mask & (adata.obs[cfg.cell_col].astype(str).values == cell)
    ctrl_counts[cell] = int(m.sum())
    if m.sum() == 0:
        raise ValueError(f"No controls for {cell}")
    ctrl_means[cell] = X_pca[m].mean(axis=0).astype(np.float32)

print("Control counts by cell type:", ctrl_counts)
pd.DataFrame([{"cell_type":k,"n_controls":v} for k,v in ctrl_counts.items()]).to_csv(
    f"{cfg.results_dir}/control_counts.csv", index=False
)

Cell types: {0: 'A549', 1: 'K562', 2: 'MCF7'}
Control counts by cell type: {'A549': 3287, 'K562': 3359, 'MCF7': 6358}


In [ ]:
X0_pca = np.stack([ctrl_means[c] for c in adata.obs[cfg.cell_col].astype(str).values]).astype(np.float32)
DELTA_pca = (X_pca - X0_pca).astype(np.float32)

def rowwise_pearson(a, b):
    vals = []
    for i in range(a.shape[0]):
        if np.std(a[i]) < 1e-8 or np.std(b[i]) < 1e-8:
            continue
        vals.append(pearsonr(a[i], b[i])[0])
    return float(np.mean(vals)) if vals else np.nan

def eval_baseline(split_name, mode="global"):
    mask = (adata.obs["_split3"].values == split_name) & (adata.obs[cfg.condition_col].astype(str).values != "control")
    idxs = np.where(mask)[0]
    train_pert = train_mask & (adata.obs[cfg.condition_col].astype(str).values != "control")
    global_mean = DELTA_pca[train_pert].mean(axis=0).astype(np.float32)

    cell_mean = {}
    for cell in sorted(adata.obs[cfg.cell_col].astype(str).unique()):
        m = train_pert & (adata.obs[cfg.cell_col].astype(str).values == cell)
        cell_mean[cell] = DELTA_pca[m].mean(axis=0).astype(np.float32) if m.sum() > 0 else global_mean.copy()

    pred, true, x0 = [], [], []
    for idx in idxs:
        cell = str(adata.obs.iloc[idx][cfg.cell_col])
        x0_i = X0_pca[idx]
        true_i = X_pca[idx]
        delta = global_mean if mode == "global" else cell_mean[cell]
        pred_i = x0_i + delta
        pred.append(pred_i); true.append(true_i); x0.append(x0_i)

    pred = np.stack(pred); true = np.stack(true); x0 = np.stack(x0)
    out = {
        "split": split_name,
        "mode": mode,
        "r2_full": float(r2_score(true.reshape(-1), pred.reshape(-1))),
        "pearson_rowmean": rowwise_pearson(true, pred),
        "mse": float(np.mean((true - pred)**2)),
    }
    for k in [20, 50]:
        vals = []
        for i in range(true.shape[0]):
            idx = np.argsort(-np.abs(true[i] - x0[i]))[:k]
            vals.append(r2_score(true[i, idx], pred[i, idx]))
        out[f"r2_top{k}"] = float(np.mean(vals))
    return out

baseline_rows = []
for sp in ["test", "ood"]:
    baseline_rows.append(eval_baseline(sp, "global"))
    baseline_rows.append(eval_baseline(sp, "cell"))

baseline_df = pd.DataFrame(baseline_rows)
print("\nBaseline metrics:")
print(baseline_df)
baseline_df.to_csv(f"{cfg.results_dir}/baseline_metrics.csv", index=False)


Baseline metrics:
  split    mode   r2_full  pearson_rowmean       mse  r2_top20  r2_top50
0  test  global  0.602405         0.762905  0.372716  0.461525  0.544917
1  test    cell  0.603489         0.763246  0.371700  0.464252  0.547430
2   ood  global  0.559701         0.738252  0.410312  0.430381  0.509394
3   ood    cell  0.563591         0.739673  0.406687  0.436299  0.514799


## Decision gates

After running, use these rules:

- **Gate A — schema/data**: all required columns present, nonzero controls for all cell types.
- **Gate B — split feasibility**: train / test / OOD counts are all nonzero and meaningful.
- **Gate C — residual feasibility**: cell-aware baseline is stronger than global baseline.
- **Gate D — hard case**: identify which cell type is weakest.

If A–C pass, move to **NB01 canonical training**.
